In [4]:
import pandas as pd

train = pd.read_csv("../data/preprocessing/train_processed_v2.csv")
test = pd.read_csv("../data/preprocessing/test_processed_v2.csv")

# 1) train/test 컬럼 구성이 완전히 같은지
assert list(train.columns) == list(test.columns), "컬럼 불일치!"

# 2) Attrition이 0/1 정수인지
print(train["Attrition"].unique())  # [0 1] 이어야 정상

# 3) 클래스 비율 (일반적으로 이직률은 소수 클래스라 Left=1 비율이 낮게 나오는 게 정상)
print(train["Attrition"].value_counts(normalize=True))

[0 1]
Attrition
0    0.524514
1    0.475486
Name: proportion, dtype: float64


In [1]:
import pandas as pd

train = pd.read_csv("../data/preprocessing/train_processed.csv")
test = pd.read_csv("../data/preprocessing/test_processed.csv")

train.head()

,Age,Gender,Years at Company,Monthly Income,Number of Promotions,Overtime,Distance from Home,Number of Dependents,Company Tenure,Remote Work,...,Job Level_Mid,Job Level_Senior,Company Size_Medium,Company Size_Small,Company Reputation_Fair,Company Reputation_Good,Company Reputation_Poor,Employee Recognition_Low,Employee Recognition_Medium,Employee Recognition_Very High
0,31,1,19,5390,2,0,22,0,89,0,...,1,0,1,0,0,0,0,0,1,0
1,59,0,4,5534,3,0,21,3,21,0,...,1,0,1,0,1,0,0,1,0,0
2,24,0,10,8159,0,0,11,3,74,0,...,1,0,1,0,0,0,1,1,0,0
3,36,0,7,3989,1,0,27,2,50,1,...,1,0,0,1,0,1,0,0,1,0
4,56,1,41,4821,0,1,71,0,68,0,...,0,1,1,0,1,0,0,0,1,0


In [ ]:
import pandas as pd

# train_df = pd.read_csv("./data/processing/train_processed.csv")

df = pd.read_csv("../data/preprocessing/train_processed.csv")


print("=" * 80)
print("1. 기본 정보")
print("=" * 80)

print(f"행 개수: {len(df):,}")
print(f"컬럼 개수: {len(df.columns):,}")

print("\n컬럼 목록:")
print(df.columns.tolist())


# ==============================================================================
# 2. 데이터 타입 확인
# ==============================================================================

print("\n" + "=" * 80)
print("2. 데이터 타입")
print("=" * 80)

print(df.dtypes)

object_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

print("\n문자열 컬럼:")
print(object_cols)

if object_cols:
    print("\n⚠ 문자열 컬럼이 남아 있습니다.")
else:
    print("\n✓ 문자열 컬럼이 없습니다.")


# ==============================================================================
# 3. 결측치 확인
# ==============================================================================

print("\n" + "=" * 80)
print("3. 결측치 확인")
print("=" * 80)

missing = df.isnull().sum()

missing = missing[missing > 0]

if missing.empty:
    print("✓ 결측치가 없습니다.")
else:
    print("⚠ 결측치가 존재합니다.")
    print(missing)


# ==============================================================================
# 4. Target 확인
# ==============================================================================

print("\n" + "=" * 80)
print("4. Attrition Target 확인")
print("=" * 80)

print(df["Attrition"].value_counts())
print()

print(df["Attrition"].value_counts(normalize=True).sort_index())


# ==============================================================================
# 5. Binary Encoding 확인
# ==============================================================================

print("\n" + "=" * 80)
print("5. Binary Encoding 확인")
print("=" * 80)

binary_columns = [
    "Gender",
    "Overtime",
    "Remote Work",
    "Leadership Opportunities",
    "Innovation Opportunities",
]

for col in binary_columns:
    if col not in df.columns:
        print(f"⚠ {col}: 컬럼 없음")
        continue

    unique_values = sorted(df[col].dropna().unique().tolist())

    print(f"{col:<30} {unique_values}")

    if set(unique_values).issubset({0, 1}):
        print("  ✓ 정상")
    else:
        print("  ⚠ 0/1 이외의 값 존재")


# ==============================================================================
# 6. Ordinal Encoding 확인
# ==============================================================================

print("\n" + "=" * 80)
print("6. Ordinal Encoding 확인")
print("=" * 80)

ordinal_columns = [
    "Job Level",
    "Company Size",
    "Company Reputation",
    "Work-Life Balance",
    "Employee Recognition",
    "Job Satisfaction",
    "Performance Rating",
    "Education Level",
]

for col in ordinal_columns:
    if col not in df.columns:
        print(f"⚠ {col}: 컬럼 없음")
        continue

    unique_values = sorted(df[col].dropna().unique().tolist())

    print(f"{col:<30} {unique_values}")


# ==============================================================================
# 7. One-Hot Encoding 확인
# ==============================================================================

print("\n" + "=" * 80)
print("7. One-Hot Encoding 확인")
print("=" * 80)

one_hot_prefixes = [
    "Job Role_",
    "Marital Status_",
]

for prefix in one_hot_prefixes:
    columns = [col for col in df.columns if col.startswith(prefix)]

    print(f"\n[{prefix}]")

    if not columns:
        print("⚠ One-Hot 컬럼 없음")
        continue

    for col in columns:
        unique_values = sorted(df[col].dropna().unique().tolist())

        print(f"{col:<35} {unique_values}")


# ==============================================================================
# 8. Feature Engineering 확인
# ==============================================================================

print("\n" + "=" * 80)
print("8. Feature Engineering 확인")
print("=" * 80)

feature_columns = [
    "Industry Experience Gap",
    "Promotion Rate",
]

for col in feature_columns:
    if col not in df.columns:
        print(f"⚠ {col}: 컬럼 없음")
        continue

    print(f"\n[{col}]")

    print(df[col].describe())

    print("결측치:", df[col].isnull().sum())


# ==============================================================================
# 9. Industry Experience Gap 검증
# ==============================================================================

print("\n" + "=" * 80)
print("9. Industry Experience Gap 계산 검증")
print("=" * 80)

if (
    "Company Tenure" in df.columns
    and "Years at Company" in df.columns
    and "Industry Experience Gap" in df.columns
):
    expected_gap = (df["Company Tenure"] - df["Years at Company"]).clip(lower=0)

    difference = (df["Industry Experience Gap"] - expected_gap).abs()

    if difference.max() == 0:
        print("✓ 모든 행의 계산이 정확합니다.")
    else:
        print("⚠ 계산이 일치하지 않는 행이 있습니다.")
        print(
            df.loc[
                difference > 0,
                [
                    "Company Tenure",
                    "Years at Company",
                    "Industry Experience Gap",
                ],
            ].head(20)
        )


# ==============================================================================
# 10. Promotion Rate 검증
# ==============================================================================

print("\n" + "=" * 80)
print("10. Promotion Rate 계산 검증")
print("=" * 80)

if (
    "Years at Company" in df.columns
    and "Number of Promotions" in df.columns
    and "Promotion Rate" in df.columns
):
    expected_rate = df["Number of Promotions"] / (df["Years at Company"] + 1)

    difference = (df["Promotion Rate"] - expected_rate).abs()

    if difference.max() < 1e-10:
        print("✓ 모든 행의 계산이 정확합니다.")
    else:
        print("⚠ 계산이 일치하지 않는 행이 있습니다.")


# ==============================================================================
# 11. 숫자가 아닌 컬럼 확인
# ==============================================================================

print("\n" + "=" * 80)
print("11. 모델 입력에 숫자가 아닌 컬럼 확인")
print("=" * 80)

non_numeric_columns = df.select_dtypes(exclude=["number"]).columns.tolist()

if not non_numeric_columns:
    print("✓ 모든 컬럼이 숫자형입니다.")
else:
    print("⚠ 숫자가 아닌 컬럼:")
    print(non_numeric_columns)


# ==============================================================================
# 12. 무한대 값 확인
# ==============================================================================

print("\n" + "=" * 80)
print("12. Infinite 값 확인")
print("=" * 80)

numeric_df = df.select_dtypes(include=["number"])

infinite_count = numeric_df.isin([float("inf"), float("-inf")]).sum().sum()

print(f"Infinite 값 개수: {infinite_count}")

if infinite_count == 0:
    print("✓ Infinite 값이 없습니다.")
else:
    print("⚠ Infinite 값이 존재합니다.")


# ==============================================================================
# 13. 중복 행 확인
# ==============================================================================

print("\n" + "=" * 80)
print("13. 중복 행 확인")
print("=" * 80)

duplicate_count = df.duplicated().sum()

print(f"중복 행 개수: {duplicate_count:,}")

if duplicate_count == 0:
    print("✓ 완전 동일한 중복 행이 없습니다.")
else:
    print("⚠ 중복 행이 존재합니다.")


# ==============================================================================
# 14. 숫자형 Feature 통계
# ==============================================================================

print("\n" + "=" * 80)
print("14. 숫자형 Feature 통계")
print("=" * 80)

print(df.describe().T.to_string())


# ==============================================================================
# 15. 최종 결과
# ==============================================================================

print("\n" + "=" * 80)
print("전처리 검증 완료")
print("=" * 80)

1. 기본 정보
행 개수: 59,598
컬럼 개수: 29

컬럼 목록:
['Age', 'Gender', 'Years at Company', 'Monthly Income', 'Work-Life Balance', 'Job Satisfaction', 'Performance Rating', 'Number of Promotions', 'Overtime', 'Distance from Home', 'Education Level', 'Number of Dependents', 'Job Level', 'Company Size', 'Company Tenure', 'Remote Work', 'Leadership Opportunities', 'Innovation Opportunities', 'Company Reputation', 'Employee Recognition', 'Attrition', 'Industry Experience Gap', 'Promotion Rate', 'Job Role_Finance', 'Job Role_Healthcare', 'Job Role_Media', 'Job Role_Technology', 'Marital Status_Married', 'Marital Status_Single']

2. 데이터 타입
Age                           int64
Gender                        int64
Years at Company              int64
Monthly Income                int64
Work-Life Balance             int64
Job Satisfaction              int64
Performance Rating            int64
Number of Promotions          int64
Overtime                      int64
Distance from Home            int64
Education Le